In [1]:
# Competition-safe setup
import os
import subprocess
import sys

WHEEL_DIR_CANDIDATES = [
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
    os.path.join(os.getcwd(), "arc_agi_3_wheels"),
]

wheel_dir = next((path for path in WHEEL_DIR_CANDIDATES if os.path.isdir(path)), None)

if wheel_dir:
    install_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--find-links",
        wheel_dir,
        "arc-agi",
        "arcengine",
        "python-dotenv",
    ]

    result = subprocess.run(install_cmd, check=False, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-2000:])
    if result.returncode != 0:
        print("Wheel install returned non-zero exit code; continuing with preinstalled packages.")
        if result.stderr:
            print(result.stderr[-2000:])
else:
    print("Wheel directory not found. Using preinstalled notebook packages.")

print(f"Python version: {sys.version.split()[0]}")
print("Setup cell complete.")

l/lib/python3.12/dist-packages (from matplotlib>=3.7.0->arc-agi) (1.4.9)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0

Python version: 3.12.12
Setup cell complete.


## Shared Imports

Run this cell once before executing the cells below.

In [2]:
import csv
import glob
import json
import os
import random
import sys
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

print("Shared imports loaded.")

Shared imports loaded.


## Data and Environment Overview

In this section we:
- read all `metadata.json` files from `environment_files`
- build a table for each game (`game_id`, tags, baseline actions)
- inspect tag distribution and save an overview CSV

In [3]:
def detect_workspace_root() -> str:
    override = os.environ.get("ARC_DATA_ROOT") or os.environ.get("ARC_WORKSPACE")
    if override:
        override = os.path.abspath(os.path.expanduser(override))
        if os.path.isdir(os.path.join(override, "environment_files")):
            return override

    current = os.path.abspath(os.getcwd())
    while True:
        if os.path.isdir(os.path.join(current, "environment_files")):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent

    kaggle_root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
    if os.path.isdir(os.path.join(kaggle_root, "environment_files")):
        return kaggle_root

    raise FileNotFoundError(
        "Cannot find project root with 'environment_files'. "
        "Set ARC_DATA_ROOT or ARC_WORKSPACE."
    )


def detect_output_root(data_root: str) -> str:
    candidates = [
        os.environ.get("ARC_OUTPUT_ROOT"),
        "/kaggle/working",
        os.getcwd(),
        data_root,
        "/tmp",
    ]

    for candidate in candidates:
        if not candidate:
            continue
        path = os.path.abspath(os.path.expanduser(candidate))
        if os.path.isdir(path) and os.access(path, os.W_OK):
            return path

    raise OSError(
        "Cannot find writable output directory. "
        "Set ARC_OUTPUT_ROOT to a writable path."
    )


WORKSPACE = detect_workspace_root()
OUTPUT_ROOT = detect_output_root(WORKSPACE)
os.environ["ARC_DATA_ROOT"] = WORKSPACE
os.environ["ARC_OUTPUT_ROOT"] = OUTPUT_ROOT
ENV_DIR = os.path.join(WORKSPACE, "environment_files")

metadata_paths = sorted(glob.glob(os.path.join(ENV_DIR, "*", "*", "metadata.json")))
records = []

for metadata_path in metadata_paths:
    with open(metadata_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    baseline_actions = data.get("baseline_actions", [])
    tags = data.get("tags", [])

    env_py_candidates = sorted(
        glob.glob(os.path.join(os.path.dirname(metadata_path), "*.py"))
    )
    env_py_path = (
        os.path.relpath(env_py_candidates[0], WORKSPACE) if env_py_candidates else ""
    )

    records.append(
        {
            "game_id": data.get("game_id", ""),
            "title": data.get("title", ""),
            "tags": tags,
            "num_levels": len(baseline_actions),
            "baseline_min": min(baseline_actions) if baseline_actions else None,
            "baseline_max": max(baseline_actions) if baseline_actions else None,
            "baseline_avg": (
                round(sum(baseline_actions) / len(baseline_actions), 2)
                if baseline_actions
                else None
            ),
            "metadata_path": os.path.relpath(metadata_path, WORKSPACE),
            "env_py_path": env_py_path,
        }
    )

games_df = pd.DataFrame(records).sort_values("game_id").reset_index(drop=True)
view_df = games_df.copy()
view_df["tags"] = view_df["tags"].apply(lambda x: ", ".join(x) if x else "no_tags")

all_tags = []
for tags in games_df["tags"]:
    if tags:
        all_tags.extend(tags)
    else:
        all_tags.append("no_tags")

tag_df = (
    pd.Series(all_tags, name="tag")
    .value_counts()
    .rename_axis("tag")
    .reset_index(name="count")
)

print(f"Workspace root: {WORKSPACE}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Total public games: {len(view_df)}")
print(f"Total public levels: {int(view_df['num_levels'].sum())}")
print(f"Average levels per game: {view_df['num_levels'].mean():.2f}")
print()
print("Tag distribution:")
display(tag_df)

print("\nPublic games overview:")
display(
    view_df[
        [
            "game_id",
            "title",
            "tags",
            "num_levels",
            "baseline_min",
            "baseline_max",
            "baseline_avg",
            "env_py_path",
        ]
    ]
)

overview_csv = os.path.join(OUTPUT_ROOT, "public_games_overview.csv")
view_df.to_csv(overview_csv, index=False)
print(f"\nOverview saved to {overview_csv}")

Workspace root: /kaggle/input/competitions/arc-prize-2026-arc-agi-3
Output root: /kaggle/working
Total public games: 25
Total public levels: 183
Average levels per game: 7.32

Tag distribution:


,tag,count
0,keyboard_click,13
1,click,7
2,keyboard,4
3,no_tags,1



Public games overview:


,game_id,title,tags,num_levels,baseline_min,baseline_max,baseline_avg,env_py_path
0,ar25-0c556536,AR25,keyboard_click,8,32,233,93.50,environment_files/ar25/0c556536/ar25.py
1,bp35-0a0ad940,BP35,keyboard_click,9,21,163,72.33,environment_files/bp35/0a0ad940/bp35.py
2,cd82-fb555c5d,CD82,keyboard_click,6,8,55,28.50,environment_files/cd82/fb555c5d/cd82.py
3,cn04-2fe56bfb,CN04,keyboard_click,6,29,300,131.50,environment_files/cn04/2fe56bfb/cn04.py
4,dc22-fdcac232,DC22,keyboard_click,6,59,578,204.67,environment_files/dc22/fdcac232/dc22.py
5,ft09-0d8bbf25,FT09,no_tags,6,12,65,34.67,environment_files/ft09/0d8bbf25/ft09.py
6,g50t-5849a774,G50T,keyboard,7,54,230,125.57,environment_files/g50t/5849a774/g50t.py
7,ka59-38d34dbb,KA59,keyboard_click,7,28,326,104.29,environment_files/ka59/38d34dbb/ka59.py
8,lf52-271a04aa,LF52,click,10,32,244,133.90,environment_files/lf52/271a04aa/lf52.py
9,lp85-305b61c3,LP85,click,8,16,159,48.50,environment_files/lp85/305b61c3/lp85.py



Overview saved to /kaggle/working/public_games_overview.csv


## Experiment Infrastructure

In this section we create a minimal experiment framework:
- fixed random seed for reproducibility
- timestamped run directory
- standard files for config and metrics logging
- helper function to append metric rows

In [4]:
# Experiment setup
seed = 42
np.random.seed(seed)
random.seed(seed)


def detect_data_root() -> str:
    explicit = (
        os.environ.get("ARC_DATA_ROOT")
        or os.environ.get("ARC_WORKSPACE_ROOT")
        or os.environ.get("ARC_WORKSPACE")
    )
    if explicit:
        explicit = os.path.abspath(os.path.expanduser(explicit))
        if os.path.isdir(os.path.join(explicit, "environment_files")):
            return explicit

    if "WORKSPACE" in globals():
        candidate = os.path.abspath(str(WORKSPACE))
        if os.path.isdir(os.path.join(candidate, "environment_files")):
            return candidate

    probe = os.path.abspath(os.getcwd())
    while True:
        if os.path.isdir(os.path.join(probe, "environment_files")):
            return probe
        parent = os.path.dirname(probe)
        if parent == probe:
            break
        probe = parent

    kaggle_root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
    if os.path.isdir(os.path.join(kaggle_root, "environment_files")):
        return kaggle_root

    raise FileNotFoundError(
        "Could not locate data root containing environment_files. "
        "Set ARC_DATA_ROOT or ARC_WORKSPACE_ROOT."
    )


def detect_output_root(data_root: str) -> str:
    candidates = [
        os.environ.get("ARC_OUTPUT_ROOT"),
        "/kaggle/working",
        os.getcwd(),
        data_root,
        "/tmp",
    ]

    for candidate in candidates:
        if not candidate:
            continue
        path = os.path.abspath(os.path.expanduser(candidate))
        if os.path.isdir(path) and os.access(path, os.W_OK):
            return path

    raise OSError(
        "Could not locate writable output root. "
        "Set ARC_OUTPUT_ROOT to a writable directory."
    )


def assert_runtime_budget(max_seconds: int = 5 * 3600 + 45 * 60) -> None:
    if max_seconds >= 6 * 3600:
        raise ValueError("Runtime budget must be below 6 hours for this competition.")


def log_metric(
    metrics_csv: str,
    phase: str,
    game_id: Optional[str],
    metric: str,
    value: float,
    source: str,
) -> None:
    current_ts = datetime.now(timezone.utc).isoformat()
    file_exists = os.path.exists(metrics_csv)
    with open(metrics_csv, "a", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["timestamp_utc", "phase", "game_id", "metric", "value", "source"],
        )
        if not file_exists:
            writer.writeheader()
        writer.writerow(
            {
                "timestamp_utc": current_ts,
                "phase": phase,
                "game_id": game_id,
                "metric": metric,
                "value": value,
                "source": source,
            }
        )


def setup_experiment_environment(data_root: str, output_root: str) -> Dict[str, str]:
    run_id = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
    run_dir = os.path.join(output_root, "runs", run_id)
    artifacts_dir = os.path.join(run_dir, "artifacts")
    logs_dir = os.path.join(run_dir, "logs")
    os.makedirs(artifacts_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    config_path = os.path.join(run_dir, "config.json")
    metrics_csv = os.path.join(logs_dir, "metrics.csv")

    config = {
        "run_id": run_id,
        "seed": seed,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "workspace_root": data_root,
        "data_root": data_root,
        "output_root": output_root,
        "internet_allowed": False,
        "max_runtime_seconds": 5 * 3600 + 45 * 60,
    }
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)

    return {
        "run_id": run_id,
        "run_dir": run_dir,
        "artifacts_dir": artifacts_dir,
        "logs_dir": logs_dir,
        "config_path": config_path,
        "metrics_csv": metrics_csv,
        "workspace_root": data_root,
        "data_root": data_root,
        "output_root": output_root,
    }


assert_runtime_budget()
data_root = detect_data_root()
output_root = detect_output_root(data_root)
os.environ["ARC_DATA_ROOT"] = data_root
os.environ["ARC_OUTPUT_ROOT"] = output_root
RUN_CONTEXT = setup_experiment_environment(data_root, output_root)

log_metric(
    RUN_CONTEXT["metrics_csv"],
    phase="phase3",
    game_id=None,
    metric="run_initialized",
    value=1.0,
    source="infrastructure",
)

print("Data root:", RUN_CONTEXT["data_root"])
print("Output root:", RUN_CONTEXT["output_root"])
print("Run initialized:", RUN_CONTEXT["run_id"])
print("Run directory:", RUN_CONTEXT["run_dir"])
print("Config path:", RUN_CONTEXT["config_path"])
print("Metrics CSV:", RUN_CONTEXT["metrics_csv"])

Data root: /kaggle/input/competitions/arc-prize-2026-arc-agi-3
Output root: /kaggle/working
Run initialized: run_20260520_020813
Run directory: /kaggle/working/runs/run_20260520_020813
Config path: /kaggle/working/runs/run_20260520_020813/config.json
Metrics CSV: /kaggle/working/runs/run_20260520_020813/logs/metrics.csv


## Baseline Agent Implementation

This section adds a lightweight baseline policy that does not require ARC runtime packages.

What it does:
- builds a baseline profile per public game from metadata
- defines a simple action policy (`RESET` + random/hybrid actions)
- runs a short dry-run simulation for a few games
- saves baseline artifacts for later comparison

In [5]:
# Load metadata prepared in Cell 6.
if "games_df" in globals():
    metadata_df = games_df.copy()
elif "overview_csv" in globals() and os.path.exists(overview_csv):
    metadata_df = pd.read_csv(overview_csv)
else:
    raise RuntimeError("Run Cell 6 first to prepare metadata.")

def normalize_tags(raw_tags) -> str:
    if isinstance(raw_tags, list):
        tags = [str(tag).strip() for tag in raw_tags if str(tag).strip()]
        return "|".join(tags) if tags else "no_tags"

    if raw_tags is None:
        return "no_tags"

    text = str(raw_tags).strip()
    if not text or text.lower() == "nan":
        return "no_tags"

    if "|" in text:
        tags = [item.strip() for item in text.split("|") if item.strip()]
        return "|".join(tags) if tags else "no_tags"

    if "," in text:
        tags = [item.strip() for item in text.split(",") if item.strip()]
        return "|".join(tags) if tags else "no_tags"

    return text


metadata_df["tags"] = metadata_df["tags"].apply(normalize_tags)

profile_columns = [
    col
    for col in ["num_levels", "baseline_min", "baseline_max", "baseline_avg"]
    if col in metadata_df.columns
]

baseline_profile = (
    metadata_df.set_index("game_id")[profile_columns].to_dict(orient="index")
    if profile_columns
    else {}
)

def baseline_choose_action(game_state: Dict, turn_idx: int, game_tags: List[str]) -> str:
    if turn_idx == 0:
        return "RESET"

    actions = game_state["available_actions"]

    if "strategy-heavy" in game_tags and "QUICK_ACTION" in actions:
        return "QUICK_ACTION"

    if random.random() < 0.15 and "RANDOM_ACTION" in actions:
        return "RANDOM_ACTION"

    return random.choice(actions)


public_game_ids = metadata_df["game_id"].dropna().astype(str).tolist()
selected_game_ids = public_game_ids[: min(5, len(public_game_ids))]

simulation_rows = []

for game_id in selected_game_ids:
    tags_value = metadata_df.loc[metadata_df["game_id"] == game_id, "tags"].iloc[0]
    tags = [] if tags_value == "no_tags" else tags_value.split("|")

    game_state = {
        "game_id": game_id,
        "score": 0,
        "available_actions": ["UP", "DOWN", "LEFT", "RIGHT", "RESET", "RANDOM_ACTION"],
    }

    for turn_idx in range(8):
        action = baseline_choose_action(game_state, turn_idx, tags)

        confidence = 0.6
        if action == "RESET":
            confidence = 0.85
        elif action == "QUICK_ACTION":
            confidence = 0.75

        simulation_rows.append(
            {
                "game_id": game_id,
                "turn": turn_idx,
                "action": action,
                "confidence": confidence,
                "tags": "|".join(tags) if tags else "no_tags",
            }
        )

        log_metric(
            RUN_CONTEXT["metrics_csv"],
            phase="phase4",
            game_id=game_id,
            metric="action_confidence",
            value=float(confidence),
            source="baseline_policy",
        )

simulation_df = pd.DataFrame(simulation_rows)
if not simulation_df.empty:
    simulation_df = simulation_df.sort_values(["game_id", "turn"]).reset_index(drop=True)

baseline_csv = os.path.join(RUN_CONTEXT["artifacts_dir"], "baseline_simulation.csv")
simulation_df.to_csv(baseline_csv, index=False)

RUN_CONTEXT["baseline_csv"] = baseline_csv

print(f"Baseline simulation rows: {len(simulation_df)}")
print(f"Saved baseline artifact: {baseline_csv}")
display(simulation_df.head(20))

Baseline simulation rows: 40
Saved baseline artifact: /kaggle/working/runs/run_20260520_020813/artifacts/baseline_simulation.csv


,game_id,turn,action,confidence,tags
0,ar25-0c556536,0,RESET,0.85,keyboard_click
1,ar25-0c556536,1,UP,0.60,keyboard_click
2,ar25-0c556536,2,DOWN,0.60,keyboard_click
3,ar25-0c556536,3,RANDOM_ACTION,0.60,keyboard_click
4,ar25-0c556536,4,RANDOM_ACTION,0.60,keyboard_click
5,ar25-0c556536,5,RESET,0.85,keyboard_click
6,ar25-0c556536,6,RANDOM_ACTION,0.60,keyboard_click
7,ar25-0c556536,7,UP,0.60,keyboard_click
8,bp35-0a0ad940,0,RESET,0.85,keyboard_click
9,bp35-0a0ad940,1,RANDOM_ACTION,0.60,keyboard_click


## Offline Generative Policy

This section adds a local, internet-free generative policy.

What it does:
- builds compact textual prompts from game state and history
- generates candidate commands using stochastic templates
- validates commands against the allowed action set
- logs prompts, actions, and confidence values as artifacts

In [6]:
def build_prompt(game_state: Dict, history_actions: List[str], memory_vector: Dict[str, float]) -> str:
    lines = [
        f"Game: {game_state['game_id']}",
        f"Turn: {game_state['turn_index']}",
        f"Score: {game_state.get('score', 0)}",
        f"Available actions: {', '.join(game_state['available_actions'])}",
        f"Memory avg reward: {memory_vector.get('avg_reward', 0.0):.3f}",
        f"Memory reset success: {memory_vector.get('reset_success', 0.0):.3f}",
    ]

    if history_actions:
        lines.append("Recent history: " + " -> ".join(history_actions[-5:]))
    else:
        lines.append("Recent history: none")

    lines.append("Produce one valid action and confidence in [0,1].")
    return "\n".join(lines)


def validate_action(action: str, available_actions: List[str]) -> str:
    if action in available_actions:
        return action
    return random.choice(available_actions)


def generate_offline_action(
    game_state: Dict,
    history_actions: List[str],
    memory_vector: Dict[str, float],
) -> Tuple[str, float, str]:
    prompt = build_prompt(game_state, history_actions, memory_vector)

    template_pool = [
        "UP",
        "DOWN",
        "LEFT",
        "RIGHT",
        "RESET",
        "RANDOM_ACTION",
    ]

    base = random.choice(template_pool)
    suffix = random.choice(["", " cautiously", " aggressively", " with minimal risk"])
    candidate = f"{base}{suffix}"

    action = validate_action(candidate, game_state["available_actions"])
    confidence = float(np.clip(np.random.normal(loc=0.68, scale=0.14), 0.05, 0.98))

    return action, confidence, prompt


def generate_action(
    game_state: Dict,
    history_actions: List[str],
    memory_vector: Dict[str, float],
) -> Tuple[str, float, str]:
    return generate_offline_action(game_state, history_actions, memory_vector)

selected_ids = simulation_df["game_id"].dropna().unique().tolist()[: min(5, len(simulation_df))]

prompt_records = []
gen_rows = []

for game_id in selected_ids:
    tags = metadata_df.loc[metadata_df["game_id"] == game_id, "tags"].iloc[0].split("|")

    memory_vector = {
        "avg_reward": float(np.random.uniform(-0.2, 0.6)),
        "reset_success": float(np.random.uniform(0.2, 0.9)),
    }

    history_actions = []

    for turn_idx in range(8):
        game_state = {
            "game_id": game_id,
            "turn_index": turn_idx,
            "score": int(np.random.randint(0, 100)),
            "available_actions": ["UP", "DOWN", "LEFT", "RIGHT", "RESET", "RANDOM_ACTION"],
        }

        action, confidence, prompt = generate_action(game_state, history_actions, memory_vector)
        history_actions.append(action)

        prompt_records.append(
            {
                "game_id": game_id,
                "turn": turn_idx,
                "prompt": prompt,
                "action": action,
                "confidence": confidence,
            }
        )

        gen_rows.append(
            {
                "game_id": game_id,
                "turn": turn_idx,
                "action": action,
                "confidence": confidence,
                "tags": "|".join(tags),
                "policy": "offline_generative",
            }
        )

        log_metric(
            RUN_CONTEXT["metrics_csv"],
            phase="phase5",
            game_id=game_id,
            metric="gen_confidence",
            value=float(confidence),
            source="offline_generator",
        )

gen_df = pd.DataFrame(gen_rows).sort_values(["game_id", "turn"]).reset_index(drop=True)

gen_actions_csv = os.path.join(RUN_CONTEXT["artifacts_dir"], "generative_actions.csv")
gen_prompts_jsonl = os.path.join(RUN_CONTEXT["artifacts_dir"], "generative_prompts.jsonl")

gen_df.to_csv(gen_actions_csv, index=False)

with open(gen_prompts_jsonl, "w") as f:
    for row in prompt_records:
        f.write(json.dumps(row) + "\n")

RUN_CONTEXT["gen_actions_csv"] = gen_actions_csv
RUN_CONTEXT["gen_prompts_jsonl"] = gen_prompts_jsonl

print(f"Generated rows: {len(gen_df)}")
print(f"Saved actions: {gen_actions_csv}")
print(f"Saved prompts: {gen_prompts_jsonl}")
display(gen_df.head(20))

Generated rows: 40
Saved actions: /kaggle/working/runs/run_20260520_020813/artifacts/generative_actions.csv
Saved prompts: /kaggle/working/runs/run_20260520_020813/artifacts/generative_prompts.jsonl


,game_id,turn,action,confidence,tags,policy
0,ar25-0c556536,0,RANDOM_ACTION,0.524337,keyboard_click,offline_generative
1,ar25-0c556536,1,RANDOM_ACTION,0.724646,keyboard_click,offline_generative
2,ar25-0c556536,2,RESET,0.901090,keyboard_click,offline_generative
3,ar25-0c556536,3,DOWN,0.787441,keyboard_click,offline_generative
4,ar25-0c556536,4,LEFT,0.598677,keyboard_click,offline_generative
5,ar25-0c556536,5,RANDOM_ACTION,0.606476,keyboard_click,offline_generative
6,ar25-0c556536,6,LEFT,0.713875,keyboard_click,offline_generative
7,ar25-0c556536,7,DOWN,0.412141,keyboard_click,offline_generative
8,bp35-0a0ad940,0,LEFT,0.644854,keyboard_click,offline_generative
9,bp35-0a0ad940,1,RESET,0.657059,keyboard_click,offline_generative


## Memory and Reflection Layer

This section introduces lightweight episodic memory:
- keeps recent state-action records in memory
- infers a weak outcome signal from score deltas
- updates per-tag and per-action priors
- stores memory snapshots and reflection logs for analysis

In [7]:
def init_game_memory(tags: List[str]) -> Dict:
    return {
        "tag_stats": {tag: {"reward_sum": 0.0, "count": 0} for tag in tags},
        "action_stats": {},
        "short_history": [],
    }


def infer_outcome_signal(prev_score: float, next_score: float) -> float:
    delta = next_score - prev_score
    return float(np.tanh(delta / 20.0))


def apply_reflection(memory: Dict, tags: List[str], action: str, reward_signal: float, turn_idx: int) -> Dict:
    for tag in tags:
        memory["tag_stats"].setdefault(tag, {"reward_sum": 0.0, "count": 0})
        memory["tag_stats"][tag]["reward_sum"] += reward_signal
        memory["tag_stats"][tag]["count"] += 1

    memory["action_stats"].setdefault(action, {"reward_sum": 0.0, "count": 0})
    memory["action_stats"][action]["reward_sum"] += reward_signal
    memory["action_stats"][action]["count"] += 1

    memory["short_history"].append(
        {
            "turn": turn_idx,
            "action": action,
            "reward_signal": reward_signal,
        }
    )
    memory["short_history"] = memory["short_history"][-10:]

    action_count = memory["action_stats"][action]["count"]
    action_reward_sum = memory["action_stats"][action]["reward_sum"]

    return {
        "action_avg_reward": action_reward_sum / max(action_count, 1),
        "history_size": len(memory["short_history"]),
    }


if "gen_df" not in globals() or gen_df.empty:
    raise RuntimeError("Generative results are not available. Execute the previous policy cell first.")

reflection_rows = []

for game_id, group in gen_df.groupby("game_id"):
    tags = metadata_df.loc[metadata_df["game_id"] == game_id, "tags"].iloc[0].split("|")
    memory = init_game_memory(tags)

    prev_score = 0.0
    ordered_group = group.sort_values("turn")

    for _, action_row in ordered_group.iterrows():
        action = action_row["action"]
        next_score = prev_score + np.random.normal(loc=2.0, scale=5.0)
        reward_signal = infer_outcome_signal(prev_score, next_score)

        summary = apply_reflection(
            memory=memory,
            tags=tags,
            action=action,
            reward_signal=reward_signal,
            turn_idx=int(action_row["turn"]),
        )

        reflection_rows.append(
            {
                "game_id": game_id,
                "turn": int(action_row["turn"]),
                "action": action,
                "reward_signal": reward_signal,
                "action_avg_reward": summary["action_avg_reward"],
                "history_size": summary["history_size"],
            }
        )

        log_metric(
            RUN_CONTEXT["metrics_csv"],
            phase="phase6",
            game_id=game_id,
            metric="reward_signal",
            value=float(reward_signal),
            source="memory_reflection",
        )

        prev_score = next_score

reflection_df = pd.DataFrame(reflection_rows).sort_values(["game_id", "turn"]).reset_index(drop=True)

memory_reflection_csv = os.path.join(RUN_CONTEXT["artifacts_dir"], "memory_reflection.csv")
reflection_df.to_csv(memory_reflection_csv, index=False)

RUN_CONTEXT["memory_reflection_csv"] = memory_reflection_csv

print(f"Reflection rows: {len(reflection_df)}")
print(f"Saved reflection artifact: {memory_reflection_csv}")
display(reflection_df.head(20))

Reflection rows: 40
Saved reflection artifact: /kaggle/working/runs/run_20260520_020813/artifacts/memory_reflection.csv


,game_id,turn,action,reward_signal,action_avg_reward,history_size
0,ar25-0c556536,0,RANDOM_ACTION,0.736631,0.736631,1
1,ar25-0c556536,1,RANDOM_ACTION,-0.129369,0.303631,2
2,ar25-0c556536,2,RESET,0.000340,0.000340,3
3,ar25-0c556536,3,DOWN,0.084581,0.084581,4
4,ar25-0c556536,4,LEFT,-0.249320,-0.249320,5
5,ar25-0c556536,5,RANDOM_ACTION,0.345763,0.317675,6
6,ar25-0c556536,6,LEFT,0.314817,0.032748,7
7,ar25-0c556536,7,DOWN,0.104369,0.094475,8
8,bp35-0a0ad940,0,LEFT,-0.033591,-0.033591,1
9,bp35-0a0ad940,1,RESET,-0.267123,-0.267123,2


## Evaluation Summary

This section computes aggregate diagnostics for baseline, generative, and memory layers.

It saves:
- compact summary table
- per-game summary table
- both files under the current run artifacts directory

In [8]:
required_frames = {
    "simulation_df": "baseline output",
    "gen_df": "generative output",
    "reflection_df": "memory output",
}
missing_frames = [name for name in required_frames if name not in globals()]
if missing_frames:
    raise RuntimeError(
        "Missing data frames: "
        + ", ".join(missing_frames)
        + ". Run Cells 10, 12, and 14 first."
    )

allowed_actions = {"UP", "DOWN", "LEFT", "RIGHT", "RESET", "RANDOM_ACTION", "QUICK_ACTION"}

baseline_eval = {
    "phase": "baseline",
    "rows": int(len(simulation_df)),
    "games": int(simulation_df["game_id"].nunique()),
    "mean_confidence": float(simulation_df["confidence"].mean()),
    "reset_share": float((simulation_df["action"] == "RESET").mean()),
    "invalid_action_share": float((~simulation_df["action"].isin(allowed_actions)).mean()),
    "mean_reward_signal": float("nan"),
}

gen_eval = {
    "phase": "generative",
    "rows": int(len(gen_df)),
    "games": int(gen_df["game_id"].nunique()),
    "mean_confidence": float(gen_df["confidence"].mean()),
    "reset_share": float((gen_df["action"] == "RESET").mean()),
    "invalid_action_share": float((~gen_df["action"].isin(allowed_actions)).mean()),
    "mean_reward_signal": float("nan"),
}

memory_eval = {
    "phase": "memory",
    "rows": int(len(reflection_df)),
    "games": int(reflection_df["game_id"].nunique()),
    "mean_confidence": float("nan"),
    "reset_share": float((reflection_df["action"] == "RESET").mean()),
    "invalid_action_share": float((~reflection_df["action"].isin(allowed_actions)).mean()),
    "mean_reward_signal": float(reflection_df["reward_signal"].mean()),
}

evaluation_summary_df = pd.DataFrame([baseline_eval, gen_eval, memory_eval])

game_baseline_df = (
    simulation_df.groupby("game_id", as_index=False)
    .agg(
        baseline_turns=("turn", "count"),
        baseline_confidence_mean=("confidence", "mean"),
        baseline_reset_share=("action", lambda s: float((s == "RESET").mean())),
    )
)

game_gen_df = (
    gen_df.groupby("game_id", as_index=False)
    .agg(
        gen_turns=("turn", "count"),
        gen_confidence_mean=("confidence", "mean"),
        gen_reset_share=("action", lambda s: float((s == "RESET").mean())),
    )
)

game_memory_df = (
    reflection_df.groupby("game_id", as_index=False)
    .agg(
        memory_turns=("turn", "count"),
        reward_signal_mean=("reward_signal", "mean"),
        reward_signal_std=("reward_signal", "std"),
    )
)

evaluation_by_game_df = (
    game_baseline_df
    .merge(game_gen_df, on="game_id", how="outer")
    .merge(game_memory_df, on="game_id", how="outer")
    .sort_values("game_id")
    .reset_index(drop=True)
)

evaluation_summary_csv = os.path.join(RUN_CONTEXT["artifacts_dir"], "evaluation_summary.csv")
evaluation_by_game_csv = os.path.join(RUN_CONTEXT["artifacts_dir"], "evaluation_by_game.csv")

evaluation_summary_df.to_csv(evaluation_summary_csv, index=False)
evaluation_by_game_df.to_csv(evaluation_by_game_csv, index=False)

RUN_CONTEXT["evaluation_summary_csv"] = evaluation_summary_csv
RUN_CONTEXT["evaluation_by_game_csv"] = evaluation_by_game_csv

log_metric(
    RUN_CONTEXT["metrics_csv"],
    phase="phase7",
    game_id=None,
    metric="gen_invalid_action_share",
    value=float(gen_eval["invalid_action_share"]),
    source="evaluation",
)

print("Evaluation summary saved:", evaluation_summary_csv)
print("Per-game summary saved:", evaluation_by_game_csv)
display(evaluation_summary_df)
display(evaluation_by_game_df.head(20))

Evaluation summary saved: /kaggle/working/runs/run_20260520_020813/artifacts/evaluation_summary.csv
Per-game summary saved: /kaggle/working/runs/run_20260520_020813/artifacts/evaluation_by_game.csv


,phase,rows,games,mean_confidence,reset_share,invalid_action_share,mean_reward_signal
0,baseline,40,5,0.662500,0.250,0.0,NaN
1,generative,40,5,0.665022,0.225,0.0,NaN
2,memory,40,5,NaN,0.225,0.0,0.07177


,game_id,baseline_turns,baseline_confidence_mean,baseline_reset_share,gen_turns,gen_confidence_mean,gen_reset_share,memory_turns,reward_signal_mean,reward_signal_std
0,ar25-0c556536,8,0.66250,0.250,8,0.658585,0.125,8,0.150977,0.310732
1,bp35-0a0ad940,8,0.69375,0.375,8,0.712206,0.250,8,0.064182,0.212273
2,cd82-fb555c5d,8,0.66250,0.250,8,0.545222,0.250,8,0.153393,0.138149
3,cn04-2fe56bfb,8,0.66250,0.250,8,0.737809,0.125,8,0.049173,0.262148
4,dc22-fdcac232,8,0.63125,0.125,8,0.671288,0.375,8,-0.058875,0.265581


## Submission Packaging

This section assembles a competition-style output table and validates mandatory columns.

Behavior:
- uses generative actions by default
- applies baseline fallback when an action is invalid
- on Kaggle, writes `/kaggle/working/submission.parquet` for direct competition upload
- keeps a diagnostics preview CSV in run artifacts
- outside Kaggle, writes parquet in artifacts when engine is available, otherwise writes CSV fallback

In [9]:
%%writefile /kaggle/working/my_agent.py
import copy
import glob
import hashlib
import importlib.util
import logging
import os
import random
import re
import time
import traceback
from collections import deque
from typing import Any, List, Optional, Tuple

import numpy as np

from agents.agent import Agent
from arcengine import ActionInput, FrameData, GameAction, GameState

logger = logging.getLogger(__name__)

# ──────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────
LLM_MODEL_PATH = "/kaggle/input/qwen2-5-0-5b-instruct/transformers/default/1"
ARC_INPUT_DIR  = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
BFS_TIMEOUT        = 60   # seconds for main BFS per level
BOOTSTRAP_TIMEOUT  = 5    # seconds per bootstrap primer
BFS_MAX_STATES     = 200_000
LLM_COOLDOWN   = 60     # minimum steps between LLM calls


# ──────────────────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────────────────
def _action_from_id(act_id: int) -> GameAction:
    _MAP = {
        0: GameAction.RESET,
        1: GameAction.ACTION1, 2: GameAction.ACTION2, 3: GameAction.ACTION3,
        4: GameAction.ACTION4, 5: GameAction.ACTION5, 6: GameAction.ACTION6,
        7: GameAction.ACTION7,
    }
    return _MAP.get(act_id, GameAction.ACTION1)


def _find_game_source(game_id: str) -> Tuple[Optional[str], Optional[str]]:
    """Return (game_py_path, class_name) for a game_id like 'ar25-0c556536'."""
    parts = game_id.split("-", 1)
    if len(parts) != 2:
        return None, None
    prefix, hash_part = parts

    path = os.path.join(
        ARC_INPUT_DIR, "environment_files", prefix, hash_part, f"{prefix}.py"
    )
    if not os.path.exists(path):
        matches = glob.glob(
            os.path.join(ARC_INPUT_DIR, "environment_files", "**", f"{prefix}.py"),
            recursive=True,
        )
        path = matches[0] if matches else None

    if not path or not os.path.exists(path):
        return None, None

    cls_name = prefix[0].upper() + prefix[1:]
    try:
        with open(path) as f:
            content = f.read(3000)
        m = re.search(r"class\s+(\w+)\s*\(", content)
        if m:
            cls_name = m.group(1)
    except Exception:
        pass

    return path, cls_name


# ──────────────────────────────────────────────────────────────────────────
# LLM ADVISOR  (Qwen-0.5B, loaded lazily)
# ──────────────────────────────────────────────────────────────────────────
class LLMAdvisor:
    """Wraps a small local language model for action guidance.

    The model is loaded on first use. If the model directory is missing or
    any import fails the advisor silently becomes a no-op.
    """

    def __init__(self, model_path: str = LLM_MODEL_PATH) -> None:
        self._path = model_path
        self._model = None
        self._tok = None
        self._device = "cpu"
        self._ok = False

    def _load(self) -> bool:
        if self._model is not None:
            return self._ok
        if not os.path.isdir(self._path):
            return False
        try:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer

            logger.info("[LLM] loading model …")
            self._tok = AutoTokenizer.from_pretrained(self._path, trust_remote_code=True)
            self._model = AutoModelForCausalLM.from_pretrained(
                self._path, torch_dtype=torch.float16, trust_remote_code=True
            )
            self._device = "cuda" if torch.cuda.is_available() else "cpu"
            self._model = self._model.to(self._device)
            self._model.eval()
            self._ok = True
            logger.info("[LLM] ready on %s", self._device)
        except Exception as e:
            logger.warning("[LLM] load failed: %s", e)
        return self._ok

    def _infer(self, prompt: str, max_new: int = 16) -> str:
        try:
            import torch
            inputs = self._tok(
                prompt, return_tensors="pt", truncation=True, max_length=512
            ).to(self._device)
            with torch.no_grad():
                out = self._model.generate(
                    **inputs,
                    max_new_tokens=max_new,
                    do_sample=False,
                    pad_token_id=self._tok.eos_token_id,
                )
            return self._tok.decode(
                out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            ).strip()
        except Exception as e:
            logger.warning("[LLM] inference error: %s", e)
            return ""

    def get_action_priorities(
        self, game_src: str, available: List[int]
    ) -> List[int]:
        """Return available action IDs sorted by LLM preference (best first)."""
        if not self._load():
            return []
        src = (game_src or "unknown game")[:500]
        avail_str = ", ".join(str(a) for a in sorted(available))
        prompt = (
            f"Game rules:\n{src}\n\n"
            f"Available actions: {avail_str}.\n"
            f"List these actions from most to least useful for solving the puzzle.\n"
            f"Reply with action numbers separated by commas only, e.g.: 6,1,2"
        )
        response = self._infer(prompt, max_new=24)
        priorities: List[int] = []
        for tok in re.findall(r"\d+", response):
            n = int(tok)
            if n in available and n not in priorities:
                priorities.append(n)
        return priorities

    def suggest_action(
        self, game_src: str, grid: np.ndarray, available: List[int]
    ) -> Optional[Tuple[int, Optional[Tuple[int, int]]]]:
        """Return (action_id, coord_or_None) or None."""
        if not self._load():
            return None
        src = (game_src or "unknown")[:400]
        nonzero = np.argwhere(grid != 0)[:15]
        cells = " ".join(f"({int(c[1])},{int(c[0])})" for c in nonzero)
        avail_str = ", ".join(str(a) for a in sorted(available))
        prompt = (
            f"Game rules:\n{src}\n\n"
            f"Active grid cells (x,y): {cells}\n"
            f"Available actions: {avail_str}.\n"
            f"Best action? Reply: number only (e.g. 1) or '6 x y' for click.\n"
        )
        response = self._infer(prompt, max_new=12)
        nums = re.findall(r"\d+", response)
        if not nums:
            return None
        act_id = int(nums[0])
        if act_id not in available:
            return None
        if act_id == 6 and len(nums) >= 3:
            return act_id, (int(nums[1]) % 64, int(nums[2]) % 64)
        return act_id, None


# ──────────────────────────────────────────────────────────────────────────
# BFS SOLVER  (adapted from ARC-AGI-3 v15)
# ──────────────────────────────────────────────────────────────────────────
class BFSSolver:
    """Offline BFS using direct game class instantiation."""

    def __init__(
        self, game_path: str, class_name: str, timeout: float = BFS_TIMEOUT
    ) -> None:
        self.game_path = game_path
        self.class_name = class_name
        self.timeout = timeout
        self.game_cls = None
        self.solutions: dict = {}
        self._failed: set = set()

    def load(self) -> bool:
        import sys
        try:
            # Use a unique module name so pickle can re-import it
            mod_name = f"_arc_game_{hashlib.md5(self.game_path.encode()).hexdigest()[:12]}"
            spec = importlib.util.spec_from_file_location(mod_name, self.game_path)
            if spec is None or spec.loader is None:
                return False
            mod = importlib.util.module_from_spec(spec)
            sys.modules[mod_name] = mod  # register so pickle can find the class
            spec.loader.exec_module(mod)
            self.game_cls = getattr(mod, self.class_name)
            logger.info("[BFS] loaded %s", self.class_name)
            return True
        except Exception as e:
            logger.warning("[BFS] load failed: %s", e)
            return False

    def _scan_actions(
        self, game, f0: np.ndarray, bg: int
    ) -> List[Tuple[int, Optional[dict]]]:
        avail = getattr(game, "_available_actions", list(range(1, 6)))
        actions: List[Tuple[int, Optional[dict]]] = []

        for a in [x for x in avail if x in (1, 2, 3, 4, 5, 7)]:
            g = copy.deepcopy(game)
            try:
                r = g.perform_action(ActionInput(id=_action_from_id(a)), raw=True)
                if r.frame:
                    # Include if ANY frame in the sequence differs (handles bounce-back animations)
                    if any(not np.array_equal(f0, np.asarray(fr, dtype=np.uint8)) for fr in r.frame):
                        actions.append((a, None))
            except Exception:
                pass

        if 6 in avail:
            t0 = time.time()
            seen: set = set()
            for y in range(0, 64, 2):
                if time.time() - t0 > 3.0:
                    break
                for x in range(0, 64, 2):
                    if f0[y, x] == bg:
                        continue
                    g = copy.deepcopy(game)
                    try:
                        ai = ActionInput(
                            id=GameAction.ACTION6,
                            data={"x": x, "y": y, "game_id": "bfs"},
                        )
                        r = g.perform_action(ai, raw=True)
                        if not r.frame:
                            continue
                        f = np.asarray(r.frame[-1], dtype=np.uint8)
                        h = hashlib.md5(f.tobytes()).hexdigest()[:12]
                        if h not in seen and not np.array_equal(f0, f):
                            seen.add(h)
                            actions.append((6, {"x": x, "y": y, "game_id": "bfs"}))
                    except Exception:
                        pass
        return actions

    def solve_level(self, level_idx: int) -> Optional[list]:
        if self.game_cls is None:
            return None
        if level_idx in self.solutions:
            return self.solutions[level_idx]
        if level_idx in self._failed:
            return None
        try:
            # Recursively ensure all prior levels are solved (uses cache after first solve)
            for i in range(level_idx):
                if i not in self.solutions:
                    sub = self.solve_level(i)
                    if sub is None:
                        logger.info("[BFS] L%d: unreachable (L%d unsolved)", level_idx, i)
                        self._failed.add(level_idx)
                        return None

            # Build game at level_idx initial state by replaying cached solutions
            # RESET on a fresh game triggers full_reset() → level 0 (correct starting point)
            game = self.game_cls()
            r = game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
            if not r or not r.frame:
                return None

            for i in range(level_idx):
                advanced = False
                for act_id, data in self.solutions[i]:
                    ai = (ActionInput(id=_action_from_id(act_id), data=data) if data
                          else ActionInput(id=_action_from_id(act_id)))
                    r = game.perform_action(ai, raw=True)
                    if (getattr(r, "levels_completed", 0) or 0) > i:
                        advanced = True
                        break
                if not advanced:
                    logger.warning("[BFS] L%d: replay failed at L%d", level_idx, i)
                    return None

            # r.frame[-1] is the initial frame of level_idx after the level transition
            if not r or not r.frame:
                return None
            f0 = np.asarray(r.frame[-1], dtype=np.uint8)
            bg = int(np.bincount(f0.flatten(), minlength=16).argmax())

            actions = self._scan_actions(game, f0, bg)
            if not actions:
                logger.info("[BFS] L%d: no effective actions", level_idx)
                self._failed.add(level_idx)
                return None

            logger.info("[BFS] L%d: BFS with %d candidates", level_idx, len(actions))
            h0 = hashlib.md5(f0.tobytes()).hexdigest()
            visited: set = {h0}
            queue: deque = deque([(game, [], 0)])
            t0 = time.time()

            while queue and len(visited) < BFS_MAX_STATES:
                if time.time() - t0 > self.timeout:
                    break
                g, hist, depth = queue.popleft()
                for act_id, data in actions:
                    try:
                        g2 = copy.deepcopy(g)
                        ai = (ActionInput(id=_action_from_id(act_id), data=data) if data
                              else ActionInput(id=_action_from_id(act_id)))
                        r2 = g2.perform_action(ai, raw=True)
                        if not r2.frame:
                            continue
                        f = np.asarray(r2.frame[-1], dtype=np.uint8)
                        h = hashlib.md5(f.tobytes()).hexdigest()
                        if h in visited:
                            continue
                        visited.add(h)
                        new_hist = hist + [(act_id, data)]
                        if (getattr(r2, "levels_completed", 0) or 0) > level_idx:
                            logger.info(
                                "[BFS] L%d: solved in %d actions (%d states)",
                                level_idx, len(new_hist), len(visited),
                            )
                            self.solutions[level_idx] = new_hist
                            return new_hist
                        if depth < 30:
                            queue.append((g2, new_hist, depth + 1))
                    except Exception:
                        continue

            logger.info("[BFS] L%d: timeout/exhausted (%.1fs, %d states)", level_idx, time.time() - t0, len(visited))

            # Bootstrap fallback: some games have "bounce-back" actions whose frame[-1]
            # equals the initial frame, so normal BFS deduplication kills them.
            # Try each candidate as a one-step primer; re-scan from the primed state.
            avail = getattr(game, "_available_actions", list(range(1, 6)))
            for primer in [x for x in avail if x in (1, 2, 3, 4, 5, 7)]:
                try:
                    gp = copy.deepcopy(game)
                    rp = gp.perform_action(ActionInput(id=_action_from_id(primer)), raw=True)
                    if not rp or not rp.frame:
                        continue
                    fp = np.asarray(rp.frame[-1], dtype=np.uint8)
                    bgp = int(np.bincount(fp.flatten(), minlength=16).argmax())
                    acts_p = self._scan_actions(gp, fp, bgp)
                    if not acts_p:
                        continue
                    logger.info("[BFS] L%d: bootstrap ACTION%d -> %d candidates", level_idx, primer, len(acts_p))
                    hp = hashlib.md5(fp.tobytes()).hexdigest()
                    vis_p: set = {h0, hp}
                    q_p: deque = deque([(gp, [(primer, None)], 0)])
                    t1 = time.time()
                    found = None
                    while q_p and len(vis_p) < BFS_MAX_STATES:
                        if time.time() - t1 > BOOTSTRAP_TIMEOUT:
                            break
                        gq, hq, dq = q_p.popleft()
                        for act_id, data in acts_p:
                            try:
                                g2 = copy.deepcopy(gq)
                                ai = (ActionInput(id=_action_from_id(act_id), data=data) if data
                                      else ActionInput(id=_action_from_id(act_id)))
                                r2 = g2.perform_action(ai, raw=True)
                                if not r2.frame:
                                    continue
                                f2 = np.asarray(r2.frame[-1], dtype=np.uint8)
                                h2 = hashlib.md5(f2.tobytes()).hexdigest()
                                if h2 in vis_p:
                                    continue
                                vis_p.add(h2)
                                nh = hq + [(act_id, data)]
                                if (getattr(r2, "levels_completed", 0) or 0) > level_idx:
                                    found = nh
                                    break
                                if dq < 30:
                                    q_p.append((g2, nh, dq + 1))
                            except Exception:
                                continue
                        if found:
                            break
                    if found:
                        logger.info("[BFS] L%d: bootstrap solved in %d actions", level_idx, len(found))
                        self.solutions[level_idx] = found
                        return found
                except Exception:
                    continue
            self._failed.add(level_idx)
            return None

        except Exception as e:
            logger.warning("[BFS] L%d error: %s", level_idx, e)
            self._failed.add(level_idx)
            return None


# ──────────────────────────────────────────────────────────────────────────
# MY AGENT
# ──────────────────────────────────────────────────────────────────────────
class MyAgent(Agent):
    """BFS + LLM-guided UCB1 agent.

    Priority per level:
      1. Replay offline BFS solution if found
      2. LLM-seeded UCB1 (priorities set from game source analysis)
      3. LLM action hint when stuck (throttled by LLM_COOLDOWN)
      4. Plain UCB1 / RESET as final fallback
    """

    MAX_ACTIONS = 10_000_000
    _C = 1.41
    _STUCK_LIMIT = 25
    _WINDOW = 12

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1_000_000) + hash(self.game_id) % 1_000_000
        random.seed(seed)
        np.random.seed(seed % (2 ** 32))

        self._simple = [
            GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3,
            GameAction.ACTION4, GameAction.ACTION5,
        ]

        # BFS setup
        game_path, cls_name = _find_game_source(self.game_id)
        if game_path and cls_name:
            self._bfs: Optional[BFSSolver] = BFSSolver(game_path, cls_name)
            self._bfs.load()
            try:
                with open(game_path) as fh:
                    self._game_src = fh.read()
            except Exception:
                self._game_src = ""
            # Pre-solve level 0 during init so the first game step does not block
            self._bfs.solve_level(0)
        else:
            self._bfs = None
            self._game_src = ""

        # LLM setup
        self._llm = LLMAdvisor()
        self._last_llm_step: int = -LLM_COOLDOWN

        # BFS replay state
        self._bfs_solution: Optional[list] = None
        self._bfs_ptr: int = 0

        # UCB1 state
        self._stuck_total: int = 0
        self._reset_level()

    def _reset_level(self) -> None:
        self._stats = [{"n": 0, "r": 0.0} for _ in range(6)]
        self._total: int = 0
        self._heatmap = np.ones((64, 64), dtype=np.float32)
        self._prev_grid: Optional[np.ndarray] = None
        self._prev_idx: Optional[int] = None
        self._prev_coord: Optional[Tuple[int, int]] = None
        self._cur_score: int = -1
        self._no_progress: int = 0
        self._recent: deque = deque(maxlen=self._WINDOW)

    # ------------------------------------------------------------------ #
    # Agent interface                                                      #
    # ------------------------------------------------------------------ #

    def is_done(self, frames: list, latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames: list, latest_frame: FrameData) -> GameAction:
        try:
            return self._act(latest_frame)
        except Exception as exc:
            logger.error("[MyAgent] step %d: %s", self.action_counter, exc)
            traceback.print_exc()
            return random.choice(self._simple)

    def _act(self, frame: FrameData) -> GameAction:
        if frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            self._prev_grid = None
            self._prev_idx = None
            return GameAction.RESET

        score = self._score_of(frame)
        grid = self._grid_of(frame)
        score_delta = max(0, score - self._cur_score) if self._cur_score >= 0 else 0

        # Record outcome of previous action BEFORE any level reset
        if self._prev_grid is not None and self._prev_idx is not None:
            changed = not np.array_equal(self._prev_grid, grid)
            self._record(self._prev_idx, self._prev_coord, changed, score_delta)
            self._no_progress = 0 if changed else self._no_progress + 1

        # New level detected — reset UCB1 state after recording
        if score != self._cur_score:
            self._reset_level()
            self._cur_score = score
            self._bfs_solution = None
            self._bfs_ptr = 0

            if self._bfs is not None:
                sol = self._bfs.solve_level(score)
                if sol:
                    self._bfs_solution = sol
                else:
                    avail_raw = frame.available_actions or list(range(1, 6))
                    self._apply_llm_priorities(avail_raw)

        # Replay BFS solution
        if self._bfs_solution and self._bfs_ptr < len(self._bfs_solution):
            act_id, data = self._bfs_solution[self._bfs_ptr]
            self._bfs_ptr += 1
            action = _action_from_id(act_id)
            if data:
                action.set_data(data)
            return action

        # Stuck or oscillating
        if self._no_progress >= self._STUCK_LIMIT or self._oscillating():
            self._stuck_total += 1
            for s in self._stats:
                s["n"] = max(0, s["n"] // 2)
            self._total = max(0, self._total // 2)
            self._no_progress = 0
            self._recent.clear()
            self._prev_grid = None
            self._prev_idx = None
            self._prev_coord = None

            # Ask LLM for a recovery action before falling back to RESET
            if self.action_counter - self._last_llm_step >= LLM_COOLDOWN:
                llm_action = self._try_llm_action(frame, grid)
                if llm_action is not None:
                    self._last_llm_step = self.action_counter
                    return llm_action

            return GameAction.RESET

        # UCB1 action selection
        idxs, has6 = self._parse_available(frame)
        ucb_scores = {i: self._ucb1(i) for i in idxs}
        if has6:
            ucb_scores[5] = self._ucb1(5)
        best = max(ucb_scores, key=lambda k: ucb_scores[k])

        if best == 5:
            x, y = self._sample_coord()
            chosen = GameAction.ACTION6
            chosen.set_data({"x": x, "y": y})
            self._prev_coord = (x, y)
        else:
            chosen = self._simple[best]
            self._prev_coord = None

        self._recent.append(best)
        self._prev_grid = grid.copy()
        self._prev_idx = best
        return chosen

    # ------------------------------------------------------------------ #
    # LLM helpers                                                          #
    # ------------------------------------------------------------------ #

    def _apply_llm_priorities(self, available_raw: List[int]) -> None:
        try:
            priorities = self._llm.get_action_priorities(self._game_src, available_raw)
            for rank, act_id in enumerate(priorities):
                idx = act_id - 1 if 1 <= act_id <= 5 else (5 if act_id == 6 else -1)
                if 0 <= idx < 6:
                    self._stats[idx]["n"] = 3
                    self._stats[idx]["r"] = max(0.0, 3.0 - rank * 0.5)
        except Exception:
            pass

    def _try_llm_action(
        self, frame: FrameData, grid: np.ndarray
    ) -> Optional[GameAction]:
        avail_raw = frame.available_actions or list(range(1, 6))
        result = self._llm.suggest_action(self._game_src, grid, avail_raw)
        if result is None:
            return None
        act_id, coord = result
        if act_id == 6:
            x, y = coord if coord else self._sample_coord()
            action = GameAction.ACTION6
            action.set_data({"x": x, "y": y})
            return action
        if 1 <= act_id <= 5:
            return self._simple[act_id - 1]
        return None

    # ------------------------------------------------------------------ #
    # UCB1 helpers                                                         #
    # ------------------------------------------------------------------ #

    def _score_of(self, frame: FrameData) -> int:
        s = getattr(frame, "score", None)
        if s is not None:
            return int(s)
        return int(getattr(frame, "levels_completed", 0) or 0)

    def _grid_of(self, frame: FrameData) -> np.ndarray:
        return np.array(frame.frame, dtype=np.int8)[-1]

    def _parse_available(self, frame: FrameData) -> Tuple[set, bool]:
        raw = frame.available_actions or []
        idxs: set = set()
        has6 = False
        for aid in raw:
            if 1 <= aid <= 5:
                idxs.add(aid - 1)
            elif aid == 6:
                has6 = True
        if not idxs and not has6:
            idxs = set(range(5))
        return idxs, has6

    def _ucb1(self, idx: int) -> float:
        s = self._stats[idx]
        if s["n"] == 0:
            return float("inf")
        return s["r"] / s["n"] + self._C * np.sqrt(np.log(max(self._total, 1)) / s["n"])

    def _record(self, idx: int, coord, changed: bool, score_delta: int) -> None:
        reward = (0.5 if changed else 0.0) + (1.0 if score_delta > 0 else 0.0)
        self._stats[idx]["n"] += 1
        self._stats[idx]["r"] += reward
        self._total += 1
        if idx == 5 and coord and changed:
            x, y = coord
            for dx in range(-3, 4):
                for dy in range(-3, 4):
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < 64 and 0 <= ny < 64:
                        w = max(0.0, 1.0 - (abs(dx) + abs(dy)) * 0.2)
                        self._heatmap[ny, nx] += reward * w

    def _oscillating(self) -> bool:
        if len(self._recent) < self._WINDOW:
            return False
        hist = list(self._recent)
        for period in (2, 3, 4):
            pattern = hist[-period:]
            repeated = (pattern * ((self._WINDOW // period) + 1))[: self._WINDOW]
            if hist == repeated:
                return True
        return False

    def _sample_coord(self) -> Tuple[int, int]:
        flat = self._heatmap.ravel()
        prob = flat / flat.sum()
        idx = int(np.random.choice(len(prob), p=prob))
        return idx % 64, idx // 64


Writing /kaggle/working/my_agent.py


In [10]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def write_local_debug_submission() -> str:
    import numpy as np
    import pandas as pd

    if "gen_df" in globals() and isinstance(gen_df, pd.DataFrame) and not gen_df.empty:
        last_turn_by_game = (
            gen_df.groupby("game_id", as_index=False)["turn"]
            .max()
            .set_index("game_id")["turn"]
            .to_dict()
        )

        rows = []
        for row in gen_df.sort_values(["game_id", "turn"]).itertuples(index=False):
            game_id = str(row.game_id)
            turn = int(row.turn)
            confidence = float(getattr(row, "confidence", 0.5))
            rows.append(
                {
                    "row_id": f"{game_id}_{turn}",
                    "game_id": game_id,
                    "end_of_game": bool(turn == int(last_turn_by_game[game_id])),
                    "score": int(np.clip(round(confidence * 100.0), 0, 100)),
                }
            )

        submission_df = pd.DataFrame(
            rows, columns=["row_id", "game_id", "end_of_game", "score"]
        )
    else:
        submission_df = pd.DataFrame(
            data=[["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        )

    target_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
    target_path = target_dir / "submission.parquet"
    try:
        submission_df.to_parquet(target_path, index=False)
    except Exception:
        target_path = target_dir / "submission.csv"
        submission_df.to_csv(target_path, index=False)
    return str(target_path)


is_rerun = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

if is_rerun:
    print("Competition rerun detected: running UCB1 adaptive agent via gateway.")

    subprocess.run(
        [
            "curl",
            "--fail",
            "--retry",
            "999",
            "--retry-all-errors",
            "--retry-delay",
            "5",
            "--retry-max-time",
            "600",
            "http://gateway:8001/api/games",
        ],
        check=True,
    )

    input_root = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3")
    source_repo = input_root / "ARC-AGI-3-Agents"
    target_repo = Path("/kaggle/working/ARC-AGI-3-Agents")

    if not source_repo.is_dir():
        raise FileNotFoundError(f"Missing competition repo at {source_repo}")

    if target_repo.exists():
        shutil.rmtree(target_repo)
    shutil.copytree(source_repo, target_repo)

    # Copy the UCB1 adaptive agent written by the %%writefile cell above
    shutil.copy(
        "/kaggle/working/my_agent.py",
        target_repo / "agents" / "templates" / "my_agent.py",
    )

    init_py = target_repo / "agents" / "__init__.py"
    init_py.write_text(
        "from typing import Type\n"
        "from dotenv import load_dotenv\n"
        "from .agent import Agent, Playback\n"
        "from .swarm import Swarm\n"
        "from .templates.random_agent import Random\n"
        "from .templates.my_agent import MyAgent\n"
        "\n"
        "load_dotenv()\n"
        "\n"
        "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n"
        '    "random": Random,\n'
        '    "myagent": MyAgent,\n'
        "}\n",
        encoding="utf-8",
    )

    env_file = target_repo / ".env"
    env_file.write_text(
        "SCHEME=http\n"
        "HOST=gateway\n"
        "PORT=8001\n"
        "ARC_API_KEY=test-key-123\n"
        "ARC_BASE_URL=http://gateway:8001/\n"
        "OPERATION_MODE=online\n"
        "ENVIRONMENTS_DIR=\n"
        "RECORDINGS_DIR=/kaggle/working/server_recording\n",
        encoding="utf-8",
    )

    run_env = os.environ.copy()
    run_env["MPLBACKEND"] = "agg"
    subprocess.run(
        [sys.executable, "main.py", "--agent", "myagent"],
        cwd=str(target_repo),
        env=run_env,
        check=True,
    )

    submission_target = Path("/kaggle/working/submission.parquet")
    if not submission_target.is_file():
        raise FileNotFoundError(
            f"Expected {submission_target} after agent run, but it was not created."
        )
    print(f"Submission created at {submission_target}")
else:
    debug_path = write_local_debug_submission()
    print(f"Non-rerun mode: wrote local debug submission to {debug_path}")


Non-rerun mode: wrote local debug submission to /kaggle/working/submission.parquet
